# Possession Play — Spielerdatenbank bauen (Kaggle)

Dieses Notebook erzeugt das vollständige `PLAYERS`-Array für das Spiel aus dem
Transfermarkt-Dataset **`davidcariboo/player-scores`** ("Football Data from
Transfermarkt"). Es führt die Logik der Repo-Skripte `data-pipeline/build_db.py`
und `data-pipeline/make_game_json.py` nacheinander aus.

Zusätzlich werden **Titel/Honours** (CL, 5 Meister, 4 Pokale, Weltmeister)
berechnet und als Feld `t` pro Spieler eingebacken.

Vereinszugehörigkeiten kommen aus zwei Quellen:
* **Einsätze** (`appearances.csv`) — präzise, reichen aber nur ~2012 zurück.
* **Transferhistorie** (`transfers.csv`) — ergänzt auch Stationen **vor 2012**.

Transfer-Vereine werden über die Klub-ID auf den **offiziellen Namen** aufgelöst;
**Jugend-/Reserve-/B-Teams** werden herausgefiltert (nur Profi-Stationen zählen).

## So benutzt du es

1. Auf https://www.kaggle.com → **Create → New Notebook** (Telefon-Verifizierung
   in den Settings ist nötig, sonst kein "Add Input").
2. Rechts **Add Input → Datasets** → **`davidcariboo/player-scores`** hinzufügen
   (Anzeigetitel: *Football Data from Transfermarkt*, Autor `davidcariboo`).
3. Dieses `.ipynb` hochladen (**File → Import Notebook**) oder den Inhalt einfügen.
4. Oben **Run All**.
5. Den **Vereins-Prüfbericht** kontrollieren: Jeder der 40 Spiel-Vereine sollte
   **genau einen** korrekten Profi-Namen matchen (keine Jugendteams, keine
   fremden Klubs). Bei Abweichung den Teilstring in `GAME_CLUBS` (Zelle 1)
   anpassen und erneut **Run All**.
6. Rechts unter **Output → /kaggle/working/** **`players_game.js`** herunterladen.

## Mount-Pfad prüfen (wichtig)

Der Datensatz mountet je nach Kaggle-Version unter
`/kaggle/input/player-scores/` **oder** `/kaggle/input/datasets/davidcariboo/player-scores/`.
`DATA` in Zelle 1 ist auf letzteren Pfad gesetzt. Stimmt der Pfad nicht, mit
`os.walk("/kaggle/input")` prüfen und `DATA` auf den Ordner mit `players.csv` setzen.

## Danach (im Repo)

Der Inhalt von `players_game.js` ersetzt `src/players.js` — siehe
`data-pipeline/README.md`. Die Ausgabe beginnt mit `export const` (1:1-Tausch).

> Caveat: Auch mit der Transferhistorie sind sehr alte / Leihstationen nicht
> garantiert lückenlos. "seit 2000" = `games.season >= 2000`.

In [ ]:
# ── Konfiguration & Helfer ─────────────────────────────────────────────
from pathlib import Path
import json, re, unicodedata
import pandas as pd

DATA = Path("/kaggle/input/datasets/davidcariboo/player-scores")
OUT  = Path("/kaggle/working")
OUT.mkdir(parents=True, exist_ok=True)

TOP5 = {
    "GB1": ("Premier League", "England"),
    "ES1": ("LaLiga",         "Spain"),
    "IT1": ("Serie A",        "Italy"),
    "L1":  ("Bundesliga",     "Germany"),
    "FR1": ("Ligue 1",        "France"),
}
SINCE = 2000

# Spielkey -> Teilstring im (offiziellen) TM-Vereinsnamen, gegen die echte
# Klubliste geprüft. LIL/OL/ROM bewusst spezifisch (Lillestrøm/Lyon-Duchère/
# Cisco Roma ausschließen).
GAME_CLUBS = {
    "FCB": "bayern", "BVB": "dortmund", "RBL": "leipzig", "B04": "leverkusen",
    "SGE": "frankfurt", "BMG": "gladbach", "VFB": "stuttgart", "WOB": "wolfsburg", "SVW": "bremen",
    "MCI": "manchester city", "MUN": "manchester united", "LIV": "liverpool", "CHE": "chelsea",
    "ARS": "arsenal football club", "TOT": "tottenham", "NEW": "newcastle", "EVE": "everton", "AVL": "aston villa",
    "BAR": "futbol club barcelona", "RMA": "real madrid", "ATM": "atletico de madrid", "SEV": "sevilla",
    "VAL": "valencia", "VIL": "villarreal",
    "JUV": "juventus", "MIL": "calcio milan", "INT": "internazionale", "NAP": "napoli", "ROM": "sportiva roma", "LAZ": "lazio",
    "PSG": "saint-germain", "ASM": "monaco", "OM": "marseille", "OL": "olympique lyon", "LIL": "lille olympique",
    "POR": "clube do porto", "SLB": "benfica", "SCP": "clube de portugal",
    "AJA": "ajax", "PSV": "philips", "FEY": "feyenoord",
}

NATION_MAP = {
    "france": "FRA", "germany": "GER", "spain": "ESP", "italy": "ITA", "netherlands": "NED",
    "belgium": "BEL", "croatia": "CRO", "england": "ENG", "portugal": "PRT", "japan": "JPN",
    "brazil": "BRA", "argentina": "ARG", "mexico": "MEX", "nigeria": "NGA",
    "cote d'ivoire": "CIV", "ivory coast": "CIV", "senegal": "SEN", "colombia": "COL",
    "united states": "USA", "usa": "USA",
}

ONLY_GAME_CLUBS = True

def norm(s: str) -> str:
    s = unicodedata.normalize("NFD", str(s)).encode("ascii", "ignore").decode()
    return s.lower().strip()

# Jugend-/Reserve-/B-Team-Erkennung (ganze Tokens im normalisierten Namen)
_YOUTH = re.compile(r"(?:^| )(u\d{1,2}|sub-?\d{1,2}|ii|iii|b|c|res|reserves?|youth|yth|jugend|castilla|mestalla|amateure?)(?:$| )")
def is_youth(name) -> bool:
    return bool(_YOUTH.search(norm(name)))

def club_key_for(tm_name) -> "str|None":
    if is_youth(tm_name):
        return None
    n = norm(tm_name)
    for key, needle in GAME_CLUBS.items():
        if needle in n:
            return key
    return None

print("Konfiguration geladen. DATA =", DATA)

In [ ]:
# ── Schritt 1: Spielerauswahl (Top-5 seit 2000) + VOLLE Vereinshistorie ──
players      = pd.read_csv(DATA / "players.csv")
clubs        = pd.read_csv(DATA / "clubs.csv")
games        = pd.read_csv(DATA / "games.csv")
appearances  = pd.read_csv(DATA / "appearances.csv")

# Spiele seit 2000: einmal nur Top-5 (Spielerauswahl), einmal alle (Vereinshistorie)
g_top5 = games[games["competition_id"].isin(TOP5) & (games["season"] >= SINCE)][["game_id"]]
g_all  = games[games["season"] >= SINCE][["game_id", "season"]]

# Spielerauswahl: jeder Spieler mit >=1 Top-5-Einsatz seit 2000
app_top5 = appearances.merge(g_top5, on="game_id", how="inner")
pids = app_top5["player_id"].unique()

# VOLLE Vereinshistorie dieser Spieler: ALLE Einsätze seit 2000 (auch Portugal/NL/Pokale)
app = appearances[appearances["player_id"].isin(pids)].merge(g_all, on="game_id", how="inner")

# players.csv – nur die ausgewählten Spieler
pl = players[players["player_id"].isin(pids)].copy()
pl = pl[["player_id", "name", "first_name", "last_name",
         "date_of_birth", "position", "country_of_citizenship"]]
pl.columns = ["tm_player_id", "name", "first_name", "last_name",
              "date_of_birth", "position", "citizenship"]
pl["last_name"] = pl["last_name"].fillna("").where(
    pl["last_name"].notna() & (pl["last_name"] != ""), pl["name"])
pl.to_csv(OUT / "players.csv", index=False)

# clubs.csv – alle Vereine aus der vollen Historie
cids = app["player_club_id"].unique()
cl = clubs[clubs["club_id"].isin(cids)].copy()
cl = cl[["club_id", "name", "domestic_competition_id"]]
cl.columns = ["tm_club_id", "name", "tm_competition_id"]
cl.to_csv(OUT / "clubs.csv", index=False)

# player_club_spells.csv – (Spieler, Verein) -> Saison-Spanne
spells = (
    app.groupby(["player_id", "player_club_id"])["season"]
       .agg(["min", "max"])
       .reset_index()
)
spells.columns = ["tm_player_id", "tm_club_id", "season_start", "season_end"]
spells.to_csv(OUT / "player_club_spells.csv", index=False)

print(f"OK  players={len(pl):>6}  clubs={len(cl):>4}  spells={len(spells):>6}  -> {OUT}/")

In [ ]:
# ── Schritt 3: Titel/Honours berechnen ──
#!/usr/bin/env python3
"""
honours.py – Berechnet Titel/Honours je Spieler für Possession Play.

Quellen (davidcariboo/player-scores): games.csv, appearances.csv, competitions.csv.

Honours (11):
  Meister (aus Punktetabelle):  MBL MPL MLL MSA ML1
  Pokal/CL (aus Finalspiel):    DFB FAC CDR CIT CL
  Weltmeister (kuratierte Kader, Namensabgleich): WM

Wichtige Datensatz-Fakten (per Probe bestätigt):
 * games reicht 2005–2025; Pokal/CL-Finals ab 2012.
 * Finals: round == "Final" (exakt! "Quarter-Finals"/"Semi-Finals" enthalten
   ebenfalls "Final"). Elfmeter sind in home/away_club_goals eingerechnet →
   Sieger immer aus den Toren ableitbar (keine Tor-Gleichstände in Finals).
 * appearances hat KEINE season-Spalte → vorher per game_id aus games joinen.
 * Coupe de France ist NICHT im Datensatz (kein domestic_cup für Frankreich).
"""
import unicodedata
import pandas as pd

# Land -> Honour-Key
TOP5_LEAGUE = {"Germany": "MBL", "England": "MPL", "Spain": "MLL", "Italy": "MSA", "France": "ML1"}
TOP5_CUP    = {"Germany": "DFB", "England": "FAC", "Spain": "CDR", "Italy": "CIT"}  # FR: keine Coupe de France
CL_COMP_ID  = "CL"


def norm(s):
    s = unicodedata.normalize("NFD", str(s)).encode("ascii", "ignore").decode()
    return s.lower().strip()


# ── Kuratierte Vereins-Ergänzungen ──────────────────────────────────────────
# Bekannte (belegte) Stationen, die im Datensatz fehlen, weil alte Transfers
# (vor ~2012) nicht enthalten sind. Spielername (wie players.name) -> Spielkeys,
# die zusätzlich gesetzt werden. Werden in der Pipeline mit den abgeleiteten
# clubs gemerged (dedupliziert). Keine erfundenen Daten — nur dokumentierte Klubs.
CLUB_OVERRIDES = {
    "Cristiano Ronaldo": ["SCP"],
    "David Beckham": ["MUN", "RMA", "MIL"],
    "Zlatan Ibrahimović": ["AJA", "JUV", "INT", "BAR"],
    "Wesley Sneijder": ["AJA", "RMA"],
    "Arjen Robben": ["PSV", "CHE", "RMA"],
    "Xabi Alonso": ["LIV"],
    "Andrea Pirlo": ["MIL", "INT"],
    "Samuel Eto'o": ["BAR", "INT"],
    "Fernando Torres": ["LIV"],
}


def detect_competitions(comps):
    """competitions.csv -> {honour_key: competition_id}. Liga via sub_type
    'first_tier', Pokal via type 'domestic_cup', CL fix 'CL'."""
    out = {}
    for _, c in comps.iterrows():
        country = c.get("country_name")
        if c.get("sub_type") == "first_tier" and country in TOP5_LEAGUE:
            out[TOP5_LEAGUE[country]] = c["competition_id"]
        if c.get("type") == "domestic_cup" and country in TOP5_CUP:
            out[TOP5_CUP[country]] = c["competition_id"]
    out["CL"] = CL_COMP_ID
    return out


def league_champion_by_season(games, comp_id):
    """{season: champion_club_id} aus 3/1/0-Punkten, Tie-Break Tordiff, dann Tore."""
    g = games[games["competition_id"] == comp_id].dropna(subset=["home_club_goals", "away_club_goals"])
    res = {}
    for season, gs in g.groupby("season"):
        pts, gd, gf = {}, {}, {}
        for _, m in gs.iterrows():
            h, a = m["home_club_id"], m["away_club_id"]
            hg, ag = int(m["home_club_goals"]), int(m["away_club_goals"])
            for c in (h, a):
                pts.setdefault(c, 0); gd.setdefault(c, 0); gf.setdefault(c, 0)
            gd[h] += hg - ag; gd[a] += ag - hg; gf[h] += hg; gf[a] += ag
            if hg > ag: pts[h] += 3
            elif ag > hg: pts[a] += 3
            else: pts[h] += 1; pts[a] += 1
        if pts:
            res[season] = max(pts, key=lambda c: (pts[c], gd[c], gf[c]))
    return res


def cup_winner_by_season(games, comp_id):
    """({season: winner_club_id}, [unentschiedene_saisons]) aus dem Finalspiel."""
    g = games[(games["competition_id"] == comp_id) & (games["round"].astype(str) == "Final")]
    res, ties = {}, []
    for season, fs in g.groupby("season"):
        m = fs.iloc[-1]  # letztes Finalspiel der Saison
        hg, ag = m["home_club_goals"], m["away_club_goals"]
        if pd.notna(hg) and pd.notna(ag) and int(hg) != int(ag):
            res[season] = m["home_club_id"] if int(hg) > int(ag) else m["away_club_id"]
        else:
            ties.append(season)  # sollte laut Probe nicht vorkommen; lieber fehlend als falsch
    return res, ties


def squad_player_ids(apps_with_season, comp_id, club_id, season):
    """player_ids mit >=1 Einsatz in comp_id für club_id in season."""
    a = apps_with_season
    sel = a[(a["competition_id"] == comp_id) & (a["player_club_id"] == club_id) & (a["season"] == season)]
    return set(sel["player_id"].unique())


# ── Weltmeister-Siegerkader (kuratiert) ──────────────────────────────────────
# Namen im Transfermarkt-Stil; Zuordnung per norm() gegen players.name.
# Fehlende/abweichende Namen führen zu Auslassung (kein Falschtreffer).
# Spieler vor ~2012 sind meist gar nicht im Spieler-Pool und matchen daher nicht.
WC_SQUADS = {
    2002: [  # Brasilien
        "Marcos", "Dida", "Rogerio Ceni", "Cafu", "Lucio", "Roque Junior", "Edmilson",
        "Roberto Carlos", "Juliano Belletti", "Anderson Polga", "Junior", "Gilberto Silva",
        "Kleberson", "Ricardinho", "Juninho Paulista", "Vampeta", "Denilson", "Rivaldo",
        "Ronaldinho", "Ronaldo", "Edilson", "Luizao", "Kaka",
    ],
    2006: [  # Italien
        "Gianluigi Buffon", "Angelo Peruzzi", "Marco Amelia", "Gianluca Zambrotta",
        "Fabio Cannavaro", "Marco Materazzi", "Alessandro Nesta", "Andrea Barzagli",
        "Fabio Grosso", "Massimo Oddo", "Cristian Zaccardo", "Andrea Pirlo",
        "Gennaro Gattuso", "Daniele De Rossi", "Mauro Camoranesi", "Simone Perrotta",
        "Francesco Totti", "Alessandro Del Piero", "Alberto Gilardino", "Luca Toni",
        "Vincenzo Iaquinta", "Filippo Inzaghi", "Simone Barone",
    ],
    2010: [  # Spanien
        "Iker Casillas", "Victor Valdes", "Pepe Reina", "Sergio Ramos", "Gerard Pique",
        "Carles Puyol", "Joan Capdevila", "Alvaro Arbeloa", "Raul Albiol", "Carlos Marchena",
        "Xavi", "Andres Iniesta", "Sergio Busquets", "Xabi Alonso", "Cesc Fabregas",
        "Javi Martinez", "David Silva", "Pedro", "Jesus Navas", "David Villa",
        "Fernando Torres", "Fernando Llorente", "Juan Mata",
    ],
    2014: [  # Deutschland
        "Manuel Neuer", "Roman Weidenfeller", "Ron-Robert Zieler", "Philipp Lahm",
        "Jerome Boateng", "Mats Hummels", "Per Mertesacker", "Benedikt Howedes",
        "Shkodran Mustafi", "Erik Durm", "Bastian Schweinsteiger", "Sami Khedira",
        "Toni Kroos", "Mesut Ozil", "Mario Gotze", "Thomas Muller", "Andre Schurrle",
        "Christoph Kramer", "Lukas Podolski", "Julian Draxler", "Miroslav Klose",
        "Kevin Grosskreutz", "Matthias Ginter",
    ],
    2018: [  # Frankreich
        "Hugo Lloris", "Steve Mandanda", "Alphonse Areola", "Benjamin Pavard",
        "Raphael Varane", "Samuel Umtiti", "Presnel Kimpembe", "Lucas Hernandez",
        "Djibril Sidibe", "Benjamin Mendy", "Paul Pogba", "N'Golo Kante", "Blaise Matuidi",
        "Corentin Tolisso", "Steven Nzonzi", "Kylian Mbappe", "Antoine Griezmann",
        "Olivier Giroud", "Ousmane Dembele", "Nabil Fekir", "Florian Thauvin",
        "Thomas Lemar", "Adil Rami",
    ],
    2022: [  # Argentinien
        "Emiliano Martinez", "Franco Armani", "Geronimo Rulli", "Nahuel Molina",
        "Cristian Romero", "Nicolas Otamendi", "Lisandro Martinez", "Nicolas Tagliafico",
        "Marcos Acuna", "Gonzalo Montiel", "German Pezzella", "Juan Foyth", "Rodrigo De Paul",
        "Alexis Mac Allister", "Enzo Fernandez", "Leandro Paredes", "Giovani Lo Celso",
        "Guido Rodriguez", "Exequiel Palacios", "Lionel Messi", "Angel Di Maria",
        "Julian Alvarez", "Lautaro Martinez", "Paulo Dybala", "Angel Correa",
        "Alejandro Gomez", "Thiago Almada",
    ],
}

# ── Treiber: Titel je Spieler berechnen -> player_titles {tm_player_id: set} ──
games_h = pd.read_csv(DATA / "games.csv")
comps_h = pd.read_csv(DATA / "competitions.csv")
apps_h  = pd.read_csv(DATA / "appearances.csv")
if "season" not in apps_h.columns:
    apps_h = apps_h.merge(games_h[["game_id", "season"]], on="game_id", how="left")

COMPS = detect_competitions(comps_h)
print("erkannte Wettbewerbe:", COMPS)

player_titles = {}
def _add(pid, key): player_titles.setdefault(pid, set()).add(key)

for key in ["MBL", "MPL", "MLL", "MSA", "ML1"]:
    cid = COMPS.get(key)
    if not cid:
        print("⚠️ keine competition_id für", key); continue
    champs = league_champion_by_season(games_h, cid)
    for season, club in champs.items():
        for pid in squad_player_ids(apps_h, cid, club, season):
            _add(pid, key)
    print(f"{key} ({cid}): {len(champs)} Meister-Saisons {sorted(champs)}")

for key in ["DFB", "FAC", "CDR", "CIT", "CL"]:
    cid = COMPS.get(key)
    if not cid:
        print("⚠️ keine competition_id für", key); continue
    winners, ties = cup_winner_by_season(games_h, cid)
    for season, club in winners.items():
        for pid in squad_player_ids(apps_h, cid, club, season):
            _add(pid, key)
    msg = f"{key} ({cid}): {len(winners)} Sieger-Saisons"
    if ties: msg += f"  ⚠️ Gleichstand-Finals (nicht vergeben): {ties}"
    print(msg)

players_h = pd.read_csv(OUT / "players.csv")
name2id = {}
for _, p in players_h.iterrows():
    name2id.setdefault(norm(p["name"]), p["tm_player_id"])
for year, names in WC_SQUADS.items():
    hits = 0
    for nm in names:
        pid = name2id.get(norm(nm))
        if pid is not None:
            _add(pid, "WM"); hits += 1
    print(f"WM {year}: {hits}/{len(names)} Namen zugeordnet")

print("Spieler mit >=1 Honour:", len(player_titles))


In [ ]:
# ── Schritt 2: auf Spiel-Vereine mappen (Einsätze + Transferhistorie) ──
players     = pd.read_csv(OUT / "players.csv")
clubs       = pd.read_csv(OUT / "clubs.csv")
spells      = pd.read_csv(OUT / "player_club_spells.csv")
transfers   = pd.read_csv(DATA / "transfers.csv")     # Transferhistorie (auch vor 2012)
full_clubs  = pd.read_csv(DATA / "clubs.csv")          # für Klub-ID -> offizieller Name
id2name = dict(zip(full_clubs["club_id"], full_clubs["name"]))

pset = set(players["tm_player_id"])
tr = transfers[transfers["player_id"].isin(pset)].copy()
# Transfer-Vereine auf offizielle Namen auflösen (sauberer als die Kurznamen)
tr["from_name"] = tr["from_club_id"].map(id2name)
tr["to_name"]   = tr["to_club_id"].map(id2name)

# tm_club_id -> Spielkey (aus den Einsatz-Vereinen)
club_key = {c["tm_club_id"]: club_key_for(c["name"]) for _, c in clubs.iterrows()}

# PRÜFBERICHT über ALLE zu mappenden Namen (Einsätze + Transfers, ohne Jugend)
all_names = set(clubs["name"].dropna())
all_names |= set(tr["from_name"].dropna()) | set(tr["to_name"].dropna())
matched_names = {}
for nm in all_names:
    k = club_key_for(nm)
    if k:
        matched_names.setdefault(k, set()).add(nm)

print("── Vereins-Zuordnung (kontrollieren!) ──")
for k in GAME_CLUBS:
    names = sorted(matched_names.get(k, []))
    flag = "" if names else "  ⚠️ KEIN TREFFER"
    print(f"  {k}: {', '.join(names) if names else '—'}{flag}")
print("────────────────────────────────────────")

# Spieler -> Set der Spielkeys: (a) aus Einsätzen, (b) aus Transfers
pclubs = {}
for _, s in spells.iterrows():
    k = club_key.get(s["tm_club_id"])
    if k:
        pclubs.setdefault(s["tm_player_id"], set()).add(k)
for _, t in tr.iterrows():
    for nm in (t["from_name"], t["to_name"]):
        k = club_key_for(nm)
        if k:
            pclubs.setdefault(t["player_id"], set()).add(k)

out = []
for _, p in players.iterrows():
    keys = sorted(set(pclubs.get(p["tm_player_id"], [])) | set(CLUB_OVERRIDES.get(p["name"], [])))
    if ONLY_GAME_CLUBS and not keys:
        continue  # für dieses Spiel irrelevant
    nat = NATION_MAP.get(norm(p.get("citizenship", "")))
    try:
        by = int(str(p["date_of_birth"])[:4])
    except Exception:
        continue
    rec = {
        "n": p["name"],
        "ln": p["last_name"] if isinstance(p["last_name"], str) and p["last_name"].strip() else p["name"],
        "by": by,
        "nat": [nat] if nat else [],
        "clubs": keys,
    }
    titles = sorted(player_titles.get(p["tm_player_id"], []))
    if titles:
        rec["t"] = titles
    out.append(rec)

out.sort(key=lambda r: r["n"])
body = ",\n  ".join(json.dumps(r, ensure_ascii=False) for r in out)
js = "export const PLAYERS = [\n  " + body + "\n];\n"
(OUT / "players_game.js").write_text(js, encoding="utf-8")
print(f"OK  {len(out)} Spieler -> {OUT}/players_game.js")
print("Download: rechts unter Output, dann Inhalt nach src/players.js übernehmen.")